Attempt use `fdc` API.

In [ ]:
from pydantic_settings import BaseSettings, SettingsConfigDict

In [ ]:
class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file='../../.env', env_file_encoding='utf-8')
    
    FDC_API_KEY: str

In [ ]:
settings = Settings()

In [ ]:
from __future__ import annotations

from typing import Any, Iterable, Literal, Sequence

import httpx
from pydantic import BaseModel, Field


# -----------------------------
# Nutrient number constants
# -----------------------------
PROTEIN_NUTRIENT_ID = 203
FAT_NUTRIENT_ID = 204
CARBS_NUTRIENT_ID = 205
ENERGY_KCAL_NUTRIENT_ID = 208

MACRO_NUTRIENT_IDS = [
    PROTEIN_NUTRIENT_ID,
    FAT_NUTRIENT_ID,
    CARBS_NUTRIENT_ID,
    ENERGY_KCAL_NUTRIENT_ID,
]


# -----------------------------
# Pydantic models
# -----------------------------
class Nutrient(BaseModel):
    id: int | None = None
    number: str | None = None
    name: str | None = None
    rank: int | None = None
    unitName: str | None = None


class FoodNutrient(BaseModel):
    id: int | None = None
    amount: float | None = None
    nutrient: Nutrient | None = None


class FoodSearchResultItem(BaseModel):
    fdcId: int
    dataType: str | None = None
    description: str
    brandOwner: str | None = None
    gtinUpc: str | None = None
    ingredients: str | None = None
    publicationDate: str | None = None
    foodNutrients: list[FoodNutrient] = Field(default_factory=list)


class FoodSearchCriteria(BaseModel):
    query: str | None = None
    dataType: list[str] | None = None
    pageSize: int | None = None
    pageNumber: int | None = None
    sortBy: str | None = None
    sortOrder: str | None = None
    brandOwner: str | None = None


class FoodSearchResponse(BaseModel):
    foodSearchCriteria: FoodSearchCriteria | None = None
    totalHits: int | None = None
    currentPage: int | None = None
    totalPages: int | None = None
    foods: list[FoodSearchResultItem] = Field(default_factory=list)


class FoodItem(BaseModel):
    fdcId: int
    dataType: str | None = None
    description: str
    brandOwner: str | None = None
    gtinUpc: str | None = None
    ingredients: str | None = None
    publicationDate: str | None = None
    servingSize: float | None = None
    servingSizeUnit: str | None = None
    householdServingFullText: str | None = None
    foodNutrients: list[FoodNutrient] = Field(default_factory=list)


class Macros(BaseModel):
    protein_g: float | None = None
    fat_g: float | None = None
    carbs_g: float | None = None
    energy_kcal: float | None = None


class FoodMacros(BaseModel):
    fdc_id: int
    description: str
    data_type: str | None = None
    brand_owner: str | None = None
    serving_size: float | None = None
    serving_size_unit: str | None = None
    household_serving_full_text: str | None = None
    macros: Macros


# -----------------------------
# Helpers
# -----------------------------
def _coerce_nutrient_number(value: str | int | None) -> int | None:
    if value is None:
        return None
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def extract_macros(food: FoodItem) -> FoodMacros:
    protein_g: float | None = None
    fat_g: float | None = None
    carbs_g: float | None = None
    energy_kcal: float | None = None

    for fn in food.foodNutrients:
        nutrient_number = _coerce_nutrient_number(fn.nutrient.number) if fn.nutrient else None
        amount = fn.amount

        if nutrient_number == PROTEIN_NUTRIENT_ID:
            protein_g = amount
        elif nutrient_number == FAT_NUTRIENT_ID:
            fat_g = amount
        elif nutrient_number == CARBS_NUTRIENT_ID:
            carbs_g = amount
        elif nutrient_number == ENERGY_KCAL_NUTRIENT_ID:
            energy_kcal = amount

    return FoodMacros(
        fdc_id=food.fdcId,
        description=food.description,
        data_type=food.dataType,
        brand_owner=food.brandOwner,
        serving_size=food.servingSize,
        serving_size_unit=food.servingSizeUnit,
        household_serving_full_text=food.householdServingFullText,
        macros=Macros(
            protein_g=protein_g,
            fat_g=fat_g,
            carbs_g=carbs_g,
            energy_kcal=energy_kcal,
        ),
    )


# -----------------------------
# Async client
# -----------------------------
FormatType = Literal["abridged", "full"]


class AsyncFoodDataCentralClient:
    BASE_URL = "https://api.nal.usda.gov/fdc/v1"

    def __init__(self, api_key: str, *, timeout: float = 20.0) -> None:
        self.client = httpx.AsyncClient(
            base_url=self.BASE_URL,
            timeout=timeout,
            params={"api_key": api_key},
        )

    async def close(self) -> None:
        await self.client.aclose()

    async def __aenter__(self) -> "AsyncFoodDataCentralClient":
        return self

    async def __aexit__(self, exc_type, exc, tb) -> None:
        await self.close()

    async def search_foods(
        self,
        query: str,
        *,
        data_type: Iterable[str] | None = None,
        page_size: int | None = None,
        page_number: int | None = None,
        sort_by: str | None = None,
        sort_order: str | None = None,
        brand_owner: str | None = None,
    ) -> FoodSearchResponse:
        params: list[tuple[str, Any]] = [("query", query)]

        if data_type:
            for dt in data_type:
                params.append(("dataType", dt))

        if page_size is not None:
            params.append(("pageSize", page_size))

        if page_number is not None:
            params.append(("pageNumber", page_number))

        if sort_by is not None:
            params.append(("sortBy", sort_by))

        if sort_order is not None:
            params.append(("sortOrder", sort_order))

        if brand_owner is not None:
            params.append(("brandOwner", brand_owner))

        response = await self.client.get("/foods/search", params=params)
        response.raise_for_status()
        return FoodSearchResponse.model_validate(response.json())

    async def get_food(
        self,
        fdc_id: int,
        *,
        format: FormatType | None = None,
        nutrients: Iterable[int] | None = None,
    ) -> FoodItem:
        params: list[tuple[str, Any]] = []

        if format is not None:
            params.append(("format", format))

        if nutrients:
            for nutrient_id in nutrients:
                params.append(("nutrients", nutrient_id))

        response = await self.client.get(f"/food/{fdc_id}", params=params)
        response.raise_for_status()
        return FoodItem.model_validate(response.json())

    async def get_foods(
        self,
        fdc_ids: Sequence[int],
        *,
        format: FormatType | None = None,
        nutrients: Iterable[int] | None = None,
    ) -> list[FoodItem]:
        params: list[tuple[str, Any]] = []

        for fdc_id in fdc_ids:
            params.append(("fdcIds", fdc_id))

        if format is not None:
            params.append(("format", format))

        if nutrients:
            for nutrient_id in nutrients:
                params.append(("nutrients", nutrient_id))

        response = await self.client.get("/foods", params=params)
        response.raise_for_status()
        payload = response.json()
        return [FoodItem.model_validate(item) for item in payload]

    # --------------------------------
    # Easy macro-focused methods
    # --------------------------------

    async def get_food_macros(self, fdc_id: int) -> FoodMacros:
        food = await self.get_food(
            fdc_id,
            nutrients=MACRO_NUTRIENT_IDS,
        )
        return extract_macros(food)

    async def get_foods_macros(self, fdc_ids: Sequence[int]) -> list[FoodMacros]:
        foods = await self.get_foods(
            fdc_ids,
            nutrients=MACRO_NUTRIENT_IDS,
        )
        return [extract_macros(food) for food in foods]

    async def search_foods_with_macros(
        self,
        query: str,
        *,
        data_type: Iterable[str] | None = None,
        page_size: int | None = 10,
        page_number: int | None = 1,
    ) -> list[FoodMacros]:
        search_result = await self.search_foods(
            query,
            data_type=data_type,
            page_size=page_size,
            page_number=page_number,
        )

        ids = [food.fdcId for food in search_result.foods]
        if not ids:
            return []

        return await self.get_foods_macros(ids)


In [ ]:
import asyncio

api_key = settings.FDC_API_KEY

async with AsyncFoodDataCentralClient(api_key) as fdc:
    # Search
    search_result = await fdc.search_foods(
        "cheddar cheese",
        page_size=10,
    )

    print("Search hits:", search_result.totalHits)
    for food in search_result.foods:
        print(food.fdcId, food.description)

    # One food -> typed full-ish response
    if search_result.foods:
        first_id = search_result.foods[0].fdcId

        food = await fdc.get_food(first_id, nutrients=MACRO_NUTRIENT_IDS)
        print("\nTyped food item:")
        print(food.model_dump())

        # One food -> easy macros only
        macros = await fdc.get_food_macros(first_id)
        print("\nEasy macros:")
        print(macros.model_dump())

    # Multiple foods -> easy macros only
    ids = [food.fdcId for food in search_result.foods[:3]]
    many_macros = await fdc.get_foods_macros(ids)

    print("\nBatch macros:")
    for item in many_macros:
        print(item.model_dump())

    # Search + hydrate macros in one step
    print("\nSearch with macros:")
    results = await fdc.search_foods_with_macros("broccoli", page_size=3)
    for item in results:
        print(item.model_dump())


Honestly, given the US nature of this dataset and the specific brands, it's not too useful.